If the ISS is close to my position and it is currently dark, then send me an email telling me to look up so I can see it

In [15]:
import requests
import datetime
import time
import smtplib

In [16]:
#position of Foley
#From this website: https://www.latlong.net/
my_lat = 30.406401
my_long = -87.682083
tzid = 'America/Chicago'

In [18]:
response = requests.get(url=f'https://api.sunrise-sunset.org/json?lat={my_lat}&lng={my_long}&formatted=0&tzid={tzid}')
data = response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [17]:
#function to determine if it's dark out or not 
#only if it's dark, you can see the ISS.

#determine sunrise and sunset timesfrom this website: https://sunrise-sunset.org/api
def darkness():
    #getting the timezone
    tzid = 'America/Chicago'
    
    #including time zone and position to get the sunrise and sunset data
    response = requests.get(url=f'https://api.sunrise-sunset.org/json?lat={my_lat}&lng={my_long}&formatted=0&tzid={tzid}')
    data = response.json()
    
    #formatting the sunset data to get only the hour 
    sunset_hour = int(data['results']['sunset'].split('T')[1].split('+')[0].split(':')[0])
    
    #get current time
    current_time = datetime.datetime.now().strftime('%H:%M:%S')
    current_hour = int(current_time.split(':')[0])
    
    #check darkness
    is_dark = ""
    if current_hour < sunset_hour:
        is_dark = False
        print('Not dark yet')
    else:
        is_dark = True
        print('Is dark outside')
        
    return is_dark

# darkness()
    
    
#function to get position of ISS
def iss_position():
    
    #using a different API to get the position of the ISS.
    #the API mentioned in the course is not working. Apparently, that's quite common
    response = requests.get(url="https://api.wheretheiss.at/v1/satellites/25544")
    data = response.json()
    
    #get only lat and long data
    iss_lat = data['latitude']
    iss_long = data['longitude']
    
    return iss_lat, iss_long

iss_lat, iss_long = iss_position()


#function to check if the ISS is within viewing range
def viewing_range():
    
    #set limits for the viewing range
    lat_upr_limit = iss_lat + 10
    lat_lwr_limit = iss_lat - 10
    
    long_upr_limit = iss_long + 10
    long_lwr_limit = iss_long - 10
    
    #Loop to determine if within viewing range
    if lat_lwr_limit <= my_lat <= lat_upr_limit and long_lwr_limit <= my_long <= long_upr_limit:
        print('ISS within view')
        within_view = True
    else:
        print('ISS outside of view')
        within_view = False
    
    return within_view

#function to send email if it's dark and the ISS is within range:

def send_alert():
    
    from_email = "krishnanrahul929@gmail.com"
    to_email = "rahulakrish@gmail.com"
    password = 'ywmvojxbfldmmmkn'

    with smtplib.SMTP("smtp.gmail.com",587) as connection: #adding this port becuase the default port 25 often fails
        connection.starttls()
        connection.login(user=from_email,password=password)
        connection.sendmail(from_addr=from_email,
                            to_addrs=to_email,
                            msg="Subject:ISS Heads UP\n\n ISS visible now. Step outside and take a look at the sky "
                           )


# get the program to run every 60 seconds
while True:
    #check it its dark and the ISS is within viewing range
    dark = darkness()
    in_range = viewing_range()
    
    #if both conditions are True, then send the email
    if dark and in_range:
        send_alert()
    #   print('Go outside and look up to see the ISS')
    else:
        print('Nope, now is not the time')
    
    #this line runs the loop every 60 seconds
    time.sleep(60)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)